# Ежедневный билдер 6 отчётов HBS

Обогащает каждый из 6 отчётов в `Отчеты/` следующими полями:
- **MEL** — категория из `MEL категории.xlsx`
- **Среднее время доставки** — из `Среднее время доставки.xlsx`
- **Резерв под проекты** — из БД, по проектам `HBS SBI` / `ВПЗ SBI`
- **Среднемесячная выдача** — из БД, по тем же проектам, за 24 мес от сегодня
- **Расчёт** — необходимое пополнение
- **Event** — тег для строк с пополнением
- **Соотношение сервисного к идеальному** — в процентах
- **Критика закупок** — AOG / Критика
- лист **Свод**

Оригиналы не трогаются. Результаты — в `Отчеты_processed/`.


## Импорты и окружение

In [ ]:
import os
import sys
import shutil
import datetime as dt
import warnings
from dateutil.relativedelta import relativedelta

import pandas as pd
from dotenv import dotenv_values
from openpyxl import load_workbook

warnings.filterwarnings('ignore')
sys.path.insert(1, '..')
sys.path.insert(1, '../..')

config = dotenv_values('../../env.env')
for key, val in config.items():
    os.environ[key] = val

os.environ['relative_sql_path'] = 'sql/analiz_sklada_HBS/'
os.environ['relative_data_path'] = 'data/analiz_sklada_za_2_goda'

from get_data_from_Amos import get_data_with_params_from_dict


## Конфигурация

In [ ]:
INPUT_DIR = 'Отчеты'
OUTPUT_DIR = 'Отчеты_processed'

MEL_FILE = 'MEL категории.xlsx'
DELIVERY_FILE = 'Среднее время доставки.xlsx'

REPORTS = {
    'A32S': dict(file='HBS_Reports_A32S.xlsx',  mel_sheet='Airbus',
                 type_rev='Airbus', use_applicability=False,
                 event_prefix='HBS', project_filter='HBS SBI'),
    'B737': dict(file='HBS_Reports_B737.xlsx',  mel_sheet='Boeing',
                 type_rev='Boeing', use_applicability=False,
                 event_prefix='HBS', project_filter='HBS SBI'),
    'EJET': dict(file='HBS_Reports_EJET.xlsx',  mel_sheet='Е170',
                 type_rev='E170', use_applicability=False,
                 event_prefix='HBS', project_filter='HBS SBI'),
    'G550': dict(file='HBS_Reports_G550.xlsx',  mel_sheet='G550',
                 type_rev='G550', use_applicability=False,
                 event_prefix='HBS', project_filter='HBS SBI'),
    'INT':  dict(file='HBS_Reports_INT.xlsx',   mel_sheet='Interior',
                 type_rev='INT', use_applicability=True,
                 event_prefix='HBS', project_filter='HBS SBI'),
    'VPZ':  dict(file='HBS_Reports_VPZ_S7.xlsx', mel_sheet=None,
                 type_rev='', use_applicability=True,
                 event_prefix='НЗ', project_filter='ВПЗ SBI'),
}

PROJECT_FILTER_VALUES = ['HBS SBI', 'ВПЗ SBI']

# Имена колонок в исходных отчётах
COL_PN = 'p/n'
COL_IDEAL = 'Идеальное количеств'
COL_OBOROT = 'В обороте'
COL_SERV = 'SERVICEABLE'
COL_ORDER_QTY = 'Order Qty'
COL_MONTHLY_ISS = 'Выдача на ВС за 1 месяц'
COL_FA_TYPE = 'FA Type'
COL_MEL = 'MEL'

NEW_COLS = [
    'Среднее время доставки',
    'Резерв под проекты',
    'Среднемесячная выдача',
    'Расчет',
    'Event',
    'Соотношение сервисного к идеальному',
    'Критика закупок',
]


## Даты периода

In [ ]:
date_y = dt.date.today()
date_y_minus_2 = date_y - relativedelta(years=2)
date_y_plus_1 = date_y + relativedelta(years=1)

print(f'Сегодня:           {date_y}')
print(f'Начало периода:    {date_y_minus_2}  (для выдачи за 24 мес)')
print(f'Окно валидности резерва: [{date_y - dt.timedelta(days=365)} … {date_y_plus_1}]')


## MEL категории

In [ ]:
def _norm(s):
    return str(s).strip().lower() if s is not None else ''

mel_lookups = {}
for key, cfg in REPORTS.items():
    if cfg['mel_sheet'] is None:
        mel_lookups[key] = None
        continue
    sheet = cfg['mel_sheet']
    df = pd.read_excel(MEL_FILE, sheet_name=sheet)
    cols = list(df.columns)
    pn_col = next((c for c in cols if _norm(c) in ('pn', 'part number', 'p/n')), None)
    mel_col = next((c for c in cols if 'mel' in _norm(c)), None)
    if pn_col is None or mel_col is None:
        raise RuntimeError(f"MEL: лист '{sheet}' имеет неожиданные колонки {cols}")
    df = df[[pn_col, mel_col]].rename(columns={pn_col: 'PN', mel_col: 'MEL'})
    df['PN'] = df['PN'].astype(str).str.strip()
    df = df.dropna(subset=['PN']).drop_duplicates(subset=['PN'], keep='first')
    mel_lookups[key] = dict(zip(df['PN'], df['MEL']))

{k: (len(v) if v else 'без MEL') for k, v in mel_lookups.items()}


## Среднее время доставки

In [ ]:
df_del = pd.read_excel(DELIVERY_FILE)
df_del.columns = [c.strip() for c in df_del.columns]
if 'partno' not in df_del.columns or 'Среднее время New' not in df_del.columns:
    raise RuntimeError(f'Файл доставки: ожидаются partno и "Среднее время New", найдены {list(df_del.columns)}')
df_del['partno'] = df_del['partno'].astype(str).str.strip()
df_del = df_del.groupby('partno', as_index=False)['Среднее время New'].mean()
delivery_lookup = dict(zip(df_del['partno'], df_del['Среднее время New']))
len(delivery_lookup)


## Загрузка данных из БД

In [ ]:
%%time
df_stock = get_data_with_params_from_dict(
    path_to_sql=f'{os.environ["relative_sql_path"]}/stock.sql',
    dict_with_str_params={'date': date_y},
)
df_stock['max_trx_date'] = df_stock['max_trx_date'].apply(pd.to_datetime)
df_stock = df_stock[~df_stock['location'].apply(lambda x: 'RECERTIFY PARTS' in str(x))].copy()
print(df_stock.shape)


In [ ]:
df_al_c = get_data_with_params_from_dict(
    path_to_sql=f'{os.environ["relative_sql_path"]}/stock_to_part_request_C.sql',
    dict_with_str_params={},
)
df_al_c.shape


In [ ]:
df_al_r = get_data_with_params_from_dict(
    path_to_sql=f'{os.environ["relative_sql_path"]}/stock_to_part_request_R.sql',
    dict_with_str_params={},
)
df_al_r.shape


In [ ]:
%%time
df_ps = get_data_with_params_from_dict(
    path_to_sql=f'{os.environ["relative_sql_path"]}/pickslip.sql',
    dict_with_str_params={'date': date_y, 'date_minus_2_y': date_y_minus_2},
)
df_ps.shape


## Маппинг projectno_i → projectno

Берём прямо из `df_stock` — там после джойна с `project` есть оба поля. Отдельный SQL не нужен.

In [ ]:
project_map = (df_stock[['projectno_i', 'projectno']]
                 .dropna(subset=['projectno_i'])
                 .drop_duplicates(subset=['projectno_i'])
                 .set_index('projectno_i')['projectno']
                 .to_dict())

needed = set(PROJECT_FILTER_VALUES)
present = set(project_map.values()) & needed
missing = needed - present
if missing:
    raise RuntimeError(f'В df_stock сегодня нет проектов {missing}. Маппинг projectno_i → projectno неполный — '
                       f'фильтр выдачи по этим проектам работать не будет.')

print(f'В маппинге {len(project_map)} проектов; нужные {needed} — на месте.')


## Резерв под проекты

Логика как в `Analiz_sklada_HBS.ipynb`:
- оставляем аллокации, где `NEW_valid_due_date` попадает в окно [today−1год, today+1год] **или** `link_type='H'`;
- джойним со стоком по `storeno_i` (расходники) / `psn` (ротейблы), оставляя только сток на проектах `HBS SBI` / `ВПЗ SBI`;
- группируем по `(partno, projectno)`.


In [ ]:
lo = date_y - dt.timedelta(days=365)
hi = date_y_plus_1

def new_valid_due_date(row):
    for col in ('pf_valid_due_date', 'wp_h_end_date'):
        v = row.get(col)
        if pd.notna(v):
            d = pd.to_datetime(v).date()
            if lo <= d <= hi:
                return v
    return None

al_c = df_al_c.copy()
al_r = df_al_r.copy()
al_c['NEW_valid_due_date'] = al_c.apply(new_valid_due_date, axis=1)
al_r['NEW_valid_due_date'] = al_r.apply(new_valid_due_date, axis=1)

al_c_f = al_c[(~al_c['NEW_valid_due_date'].isna()) | (al_c['link_type'] == 'H')]
al_r_f = al_r[(~al_r['NEW_valid_due_date'].isna()) | (al_r['link_type'] == 'H')]

stock_proj = df_stock[df_stock['projectno'].isin(PROJECT_FILTER_VALUES)].copy()
if 'storeno_i' not in stock_proj.columns:
    raise RuntimeError('В df_stock нет storeno_i — не сджойнить аллокации C')
if 'psn' not in stock_proj.columns:
    raise RuntimeError('В df_stock нет psn — не сджойнить аллокации R')

sc = (stock_proj[['storeno_i', 'projectno']]
      .dropna(subset=['storeno_i'])
      .drop_duplicates(subset=['storeno_i']))
sr = (stock_proj[['psn', 'projectno']]
      .dropna(subset=['psn'])
      .drop_duplicates(subset=['psn']))

al_c_m = al_c_f.merge(sc, left_on='consumables_storeno_i', right_on='storeno_i', how='inner')
al_c_g = al_c_m.groupby(['consumables_partno', 'projectno'], as_index=False)['al_qty'].sum()
al_c_g.columns = ['partno', 'projectno', 'al_qty']

al_r_m = al_r_f.merge(sr, left_on='rotables_psn', right_on='psn', how='inner')
al_r_g = al_r_m.groupby(['rotables_partno', 'projectno'], as_index=False)['al_qty'].sum()
al_r_g.columns = ['partno', 'projectno', 'al_qty']

al_all = pd.concat([al_c_g, al_r_g], ignore_index=True)
al_all = al_all.groupby(['partno', 'projectno'], as_index=False)['al_qty'].sum()

reserve_lookup = {(r['partno'], r['projectno']): float(r['al_qty']) for _, r in al_all.iterrows()}
print(f'Записей в reserve_lookup: {len(reserve_lookup)}')
al_all.head()


## Среднемесячная выдача

Сумма выдачи за 24 мес / 24, по проектам `HBS SBI` / `ВПЗ SBI`. Маппинг `projectno_i → projectno` — из `df_stock` (см. выше). Без альтернатив.


In [ ]:
df_ps_w = df_ps.copy()
df_ps_w['qty'] = df_ps_w['qty_booked'] - df_ps_w['qty_canceled']
df_ps_w['projectno'] = df_ps_w['projectno_i'].map(project_map)
df_ps_w = df_ps_w[df_ps_w['projectno'].isin(PROJECT_FILTER_VALUES)]
ps_grp = df_ps_w.groupby(['partno', 'projectno'], as_index=False)['qty'].sum()
ps_grp['avg_monthly'] = ps_grp['qty'] / 24.0

consumption_lookup = {(r['partno'], r['projectno']): float(r['avg_monthly']) for _, r in ps_grp.iterrows()}
print(f'Записей в consumption_lookup: {len(consumption_lookup)}')
ps_grp.head()


## Функции расчёта по строке

In [ ]:
def _num(x):
    if x is None:
        return 0.0
    try:
        if pd.isna(x):
            return 0.0
    except (TypeError, ValueError):
        pass
    try:
        return float(x)
    except (TypeError, ValueError):
        return 0.0


def compute_calc(row):
    """Кейс 1: Идеальное > SERVICEABLE - Резерв  →  Идеальное - SERVICEABLE + Резерв
    Кейс 2: |Среднемес| > |Выдача за мес|  →  max(|Среднемес|-|Выдача за мес|+Резерв, Идеальное-SERVICEABLE)
    Если оба — max(v1, v2). Если ни одного / итог ≤ 0 — пусто."""
    ideal = _num(row[COL_IDEAL])
    serv = _num(row[COL_SERV])
    reserve = _num(row['Резерв под проекты'])
    avg_month = abs(_num(row['Среднемесячная выдача']))
    monthly_iss = abs(_num(row[COL_MONTHLY_ISS]))

    case1 = ideal > (serv - reserve)
    case2 = avg_month > monthly_iss

    v1 = ideal - serv + reserve if case1 else None
    v2 = max(avg_month - monthly_iss + reserve, ideal - serv) if case2 else None

    candidates = [v for v in (v1, v2) if v is not None]
    if not candidates:
        return ''
    val = max(candidates)
    return val if val > 0 else ''


def compute_event(row, cfg):
    parts = [cfg['event_prefix'], cfg['type_rev']]
    if cfg['use_applicability']:
        fa = row.get(COL_FA_TYPE, '')
        parts.append('' if pd.isna(fa) else str(fa))
    parts += ['ПОКУПКА', str(row[COL_PN]), 'причина']
    return '_'.join(parts)


def compute_ratio(row, key):
    ideal = _num(row[COL_IDEAL])
    serv = _num(row[COL_SERV])
    if ideal == 0:
        raise RuntimeError(f'[{key}] Идеальное количество = 0 для PN={row[COL_PN]}')
    return serv / ideal * 100.0


def compute_criticality(row):
    ratio = row['Соотношение сервисного к идеальному']
    order_qty = _num(row[COL_ORDER_QTY])
    if ratio == 0 and order_qty > 0:
        mel = str(row.get(COL_MEL, '') or '').upper().replace(' ', '')
        if 'AOG' in mel or 'NOGO' in mel:
            return 'AOG'
        return 'Критика'
    return ''


## Обработка одного отчёта + Свод

In [ ]:
def process_report(key, cfg):
    print(f'=== {key}: {cfg["file"]} ===')
    in_path = os.path.join(INPUT_DIR, cfg['file'])
    df = pd.read_excel(in_path, sheet_name='HBS_AMOS', header=1)
    df[COL_PN] = df[COL_PN].astype(str).str.strip()

    # 1. MEL
    if mel_lookups[key] is None:
        df[COL_MEL] = ''
    else:
        missing = sorted({pn for pn in df[COL_PN] if pn not in mel_lookups[key]})
        if missing:
            raise RuntimeError(
                f"[{key}] в MEL '{cfg['mel_sheet']}' нет {len(missing)} p/n. "
                f"Первые 10: {missing[:10]}"
            )
        df[COL_MEL] = df[COL_PN].map(mel_lookups[key])

    # 2. Среднее время доставки
    df['Среднее время доставки'] = df[COL_PN].map(delivery_lookup)

    # 3-4. Резерв и среднемесячная выдача
    proj = cfg['project_filter']
    df['Резерв под проекты'] = df[COL_PN].map(lambda pn: reserve_lookup.get((pn, proj), 0))
    df['Среднемесячная выдача'] = df[COL_PN].map(lambda pn: consumption_lookup.get((pn, proj), 0))

    # 5. Расчёт
    df['Расчет'] = df.apply(compute_calc, axis=1)

    # 6. Event — только если Расчёт > 0
    df['Event'] = df.apply(
        lambda r: compute_event(r, cfg)
        if isinstance(r['Расчет'], (int, float)) and r['Расчет'] > 0
        else '',
        axis=1,
    )

    # 7. Соотношение сервис / идеал
    df['Соотношение сервисного к идеальному'] = df.apply(lambda r: compute_ratio(r, key), axis=1)

    # 8. Критика закупок
    df['Критика закупок'] = df.apply(compute_criticality, axis=1)

    return df


def compute_summary(df):
    total = len(df)
    servable = int((df[COL_SERV].fillna(0) > 0).sum())
    no_oborot = int((df[COL_OBOROT].fillna(0) == 0).sum())
    in_oborot_no_serv = total - servable - no_oborot
    def pct(x):
        return x / total * 100 if total else 0
    return [
        ('Кол-во HBS', total, pct(total)),
        ('Годные', servable, pct(servable)),
        ('Отсутствующие в обороте', no_oborot, pct(no_oborot)),
        ('Имеющиеся в обороте и отсутствующие в годном состоянии',
         in_oborot_no_serv, pct(in_oborot_no_serv)),
    ]


## Запись результата (с сохранением форматирования)

In [ ]:
def _is_nan(v):
    try:
        return pd.isna(v)
    except (TypeError, ValueError):
        return False


def _to_cell(v):
    if v is None or v == '' or _is_nan(v):
        return None
    if isinstance(v, float):
        return round(v, 4)
    return v


def write_report(in_path, out_path, df_processed, summary):
    os.makedirs(os.path.dirname(out_path) or '.', exist_ok=True)
    shutil.copy(in_path, out_path)

    wb = load_workbook(out_path)
    ws = wb['HBS_AMOS']

    HEADER_ROW = 2          # 1-я строка — 'Amos HBS', 2-я — шапка
    DATA_START_ROW = 3

    header_to_col = {}
    for cell in ws[HEADER_ROW]:
        if cell.value is not None:
            header_to_col[str(cell.value).strip()] = cell.column

    mel_col = header_to_col.get(COL_MEL)
    if mel_col is None:
        raise RuntimeError(f"{in_path}: нет колонки '{COL_MEL}' в шапке")

    # MEL — в существующую колонку
    for i, v in enumerate(df_processed[COL_MEL].tolist()):
        ws.cell(row=DATA_START_ROW + i, column=mel_col, value=_to_cell(v))

    # Новые колонки — справа
    last_col = ws.max_column
    for j, col_name in enumerate(NEW_COLS):
        col_idx = last_col + 1 + j
        ws.cell(row=HEADER_ROW, column=col_idx, value=col_name)
        for i, v in enumerate(df_processed[col_name].tolist()):
            ws.cell(row=DATA_START_ROW + i, column=col_idx, value=_to_cell(v))

    # Свод
    if 'Свод' in wb.sheetnames:
        del wb['Свод']
    ws_s = wb.create_sheet('Свод')
    ws_s.cell(row=1, column=1, value='Показатель')
    ws_s.cell(row=1, column=2, value='Кол-во')
    ws_s.cell(row=1, column=3, value='Доля, %')
    for i, (label, count, share) in enumerate(summary, start=2):
        ws_s.cell(row=i, column=1, value=label)
        ws_s.cell(row=i, column=2, value=count)
        ws_s.cell(row=i, column=3, value=round(share, 2))

    wb.save(out_path)


## Прогон по всем 6 отчётам

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

processed = {}
for key, cfg in REPORTS.items():
    df_p = process_report(key, cfg)
    summary = compute_summary(df_p)
    in_path = os.path.join(INPUT_DIR, cfg['file'])
    out_path = os.path.join(OUTPUT_DIR, cfg['file'])
    write_report(in_path, out_path, df_p, summary)
    processed[key] = df_p
    print(f'  → {out_path}: всего {summary[0][1]}, '
          f'годных {summary[1][1]}, без оборота {summary[2][1]}, '
          f'есть в обороте, нет в годном {summary[3][1]}')

print(f'\n✓ Готово. Результаты в ./{OUTPUT_DIR}/')


## Быстрая проверка результата (опционально)

Просмотр первых 10 строк с непустым Расчётом по любому отчёту:


In [ ]:
key = 'A32S'
df_check = processed[key]
df_check[df_check['Расчет'] != ''][[COL_PN, COL_IDEAL, COL_SERV, 'Резерв под проекты',
                                     'Среднемесячная выдача', COL_MONTHLY_ISS,
                                     'Расчет', 'Event', 'Соотношение сервисного к идеальному',
                                     'Критика закупок']].head(10)
